# Running Palace Simulations

[Palace](https://awslabs.github.io/palace/) is an open-source 3D electromagnetic simulator supporting eigenmode, driven (S-parameter), and electrostatic simulations. This notebook demonstrates using the `gsim.palace` API to run a driven simulation on custom RF components—specifically, the new symmetric transformer generated using `gdsfactory`.

**Requirements:**
- `gsim` with Palace backend
- `circulax` (for lumped-element modeling and data fitting)

In [ ]:
import gdsfactory as gf
from gdsfactory.components.analog.transformers import symmetric_transformer

gf.gpdk.PDK.activate()

### Transformer layout

In [ ]:
symmetric_transformer().plot()

In [ ]:
cc = symmetric_transformer(add_pgs=True)
cc.plot()

### Configure and run simulation with DrivenSim

In [ ]:
from gsim.palace import DrivenSim

# Create simulation object
sim = DrivenSim()

# Set output directory
sim.set_output_dir("./palace-sim-symmetric_transformer")

# Set the component geometry
sim.set_geometry(cc)

# Configure layer stack from active PDK
sim.set_stack(substrate_thickness=180.0, include_substrate=True)

# Configure ports
sim.add_port(
    "P1+", from_layer="metal1", to_layer="metal3", geometry="interlayer", excited=True
)
sim.add_port(
    "P1-", from_layer="metal1", to_layer="metal3", geometry="interlayer", excited=True
)
sim.add_port(
    "P2+", from_layer="metal1", to_layer="metal3", geometry="interlayer", excited=True
)
sim.add_port(
    "P2-", from_layer="metal1", to_layer="metal3", geometry="interlayer", excited=True
)

# Configure driven simulation (frequency sweep for S-parameters)
sim.set_driven(fmin=10e9, fmax=150e9, num_points=50)

# Validate configuration
print(sim.validate_config())

In [ ]:
# Generate mesh (presets: "coarse", "default", "fine")
sim.set_airbox(margin_x=50, margin_y=50, z_above=50, z_below=5)
sim.mesh(preset="default", refined_mesh_size=1.5)
sim.write_config()

In [ ]:
sim.plot_mesh(show_groups=["metal", "via", "P"])

In [ ]:
sim.plot_mesh(
    style="solid",
    transparent_groups=["air__None", "sio2__None", "air__sio2"],
)

### Run simulation on cloud

In [ ]:
# Run simulation on GDSFactory+ cloud
results = sim.run()

In [ ]:
results.plot_interactive()

In [ ]:
results.plot_interactive(phase=True)

In [ ]:
results.plot()